In [ ]:
import networkx as nx
import random
import warnings
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter, defaultdict
from datetime import datetime, timedelta
import os
import joblib
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')

 ## DATASET + GRAFO CON AUTOMA DETERMINISTICO


In [ ]:
interval_labeling = pd.read_csv("../data/csv/intervalli_glucosio.csv", delimiter=',')
df = pd.DataFrame(interval_labeling)
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])
df["duration"] = ((df["end_time"] - df["start_time"]).dt.total_seconds() / 60).round(2)
arr = []
for index, row in df.iterrows():
    arr.append((row['label'], row['duration']))

seq = arr

G = nx.DiGraph()

for i in range(len(seq) - 1):
    src, dur_src = seq[i]
    dst, dur_dst = seq[i + 1]
    if G.has_edge(src, dst):
        G[src][dst]["count"] += 1
        G[src][dst]["durations"].append(dur_dst)
    else:
        G.add_edge(src, dst, count=1, durations=[dur_dst])


## EVENTI SU AUTOMA DETERMINISTICO

In [ ]:
def most_n_frequent_events(seq, n=3):
    ngrams = defaultdict(lambda: {'count': 0, 'durations': []})
    for i in range(len(seq) - n + 1):
        labels = tuple([s[0] for s in seq[i:i + n]])
        #durations = [s[1] for s in seq[i:i+n]]
        ngrams[labels]['count'] += 1
        #ngrams[labels]['durations'].append(durations)
    return ngrams


# --- Calcolo media durate per ogni transizione ---
for u, v, data in G.edges(data=True):
    data["mean_duration"] = sum(data["durations"]) / len(data["durations"])

# --- Calcolo delle probabilità di transizione ---
for u in G.nodes():
    total = sum(G[u][v]["count"] for v in G.successors(u))
    for v in G.successors(u):
        if total > 0:
            G[u][v]["prob"] = G[u][v]["count"] / total
        else:
            G[u][v]["prob"] = 0.0  # fallback se nessuna uscita

# --- Stampa riepilogativa delle transizioni ---
print("\n Modello probabilistico delle transizioni:\n")
for u in G.nodes():
    successors = list(G.successors(u))
    if not successors:
        print(f"{u} → [nessuna uscita]")
        continue
    print(f"\nDa stato '{u}' posso passare a:")
    for v in successors:
        prob = G[u][v]["prob"]
        mean_dur = round(G[u][v]["mean_duration"], 2)
        count = G[u][v]["count"]
        print(f"  ↳ {v:<16} | Probabilità: {prob:.2f} | Count: {count:<3} | Durata media: {mean_dur} min")

# --- Eventi più frequenti ---
events = most_n_frequent_events(seq, 4)
print("\n Esempio di 4 pattern più frequenti:")
for k, v in sorted(events.items(), key=lambda x: x[1]['count'], reverse=True)[:10]:
    print(f"{k} | count={v['count']}")

## CREAZIONE SEQUENZE REALISTICHE E RANDOMICHE

In [ ]:
def simulate_random_sequence_realistic(
    G,
    total_minutes=10000,
    balanced=False,
    forced_length=None,
    noise_mode=True,
    borderline=False,
    forced_target=None
):
    """
    Generatore realistico controllato.

    Parametri:
    - balanced=True → target distribuzione quasi uniforme per test
    - forced_target="verde"/"giallo"/"rosso" → forza un target specifico
    - borderline=True → valori vicini alle soglie
    - forced_length=N → lunghezza sequenza in minuti
    - noise_mode=True → rumore tipo CGM

    Return:
        seq, true_color, gmi, avg_gluc, gmi_color
    """

    import numpy as np
    import random
    from collections import defaultdict

    if forced_length is not None:
        total_minutes = forced_length

    # ======================================================
    # 1. Scelta target colore
    # ======================================================
    if forced_target is not None:
        target_color = forced_target
    elif balanced:
        target_color = random.choices(
            ["verde", "giallo", "rosso"],
            weights=[1/3, 1/3, 1/3],
            k=1
        )[0]
    else:
        # Distribuzione fissa: 60% verde, 30% giallo, 10% rosso
        target_color = random.choices(
            ["verde", "giallo", "rosso"],
            weights=[0.6, 0.3, 0.1],
            k=1
        )[0]

    # ======================================================
    # 2. Range glicemici coerenti col target
    # ======================================================
    if borderline:
        glucose_ranges_mode = {
            "verde": {"extremely_low": (60, 70), "low": (70, 85), "normal": (90, 150), "high": (150, 170), "extremely_high": (170, 185)},
            "giallo": {"extremely_low": (55, 70), "low": (70, 85), "normal": (100, 170), "high": (180, 210), "extremely_high": (210, 230)},
            "rosso": {"extremely_low": (40, 55), "low": (55, 75), "normal": (120, 180), "high": (200, 260), "extremely_high": (260, 350)}
        }
    else:
        glucose_ranges_mode = {
            "verde": {"extremely_low": (65, 75), "low": (75, 90), "normal": (90, 150), "high": (150, 180), "extremely_high": (180, 220)},
            "giallo": {"extremely_low": (55, 70), "low": (70, 85), "normal": (100, 170), "high": (180, 230), "extremely_high": (230, 260)},
            "rosso": {"extremely_low": (40, 55), "low": (55, 75), "normal": (120, 180), "high": (200, 300), "extremely_high": (300, 380)}
        }
    glucose_ranges = glucose_ranges_mode[target_color]

    # ======================================================
    # 3. Probabilità base transizioni coerenti
    # ======================================================
    if target_color == "verde":
        base_probs = {"normal": 0.85, "high": 0.08, "low": 0.06, "extreme": 0.01}
    elif target_color == "giallo":
        base_probs = {"normal": 0.45, "high": 0.28, "low": 0.22, "extreme": 0.05}
    else:
        base_probs = {"normal": 0.15, "high": 0.45, "low": 0.3, "extreme": 0.1}

    # ======================================================
    # 4. Generazione sequenza
    # ======================================================
    seq = []
    state = "normal"
    elapsed = 0
    counts = defaultdict(int)
    total_gluc_area = 0
    total_gluc_time = 0

    while elapsed < total_minutes:
        successors = list(G.successors(state))
        if not successors:
            state = random.choice(list(G.nodes))
            successors = list(G.successors(state))
            if not successors:
                continue

        succ_probs = []
        for s in successors:
            if s == "normal":
                base = base_probs["normal"]
            elif s in ["high", "low"]:
                base = base_probs["high"] if s=="high" else base_probs["low"]
            else:
                base = base_probs["extreme"]
            base *= 1 / (1 + counts[s]/50)
            succ_probs.append(base)
        succ_probs = np.array(succ_probs)
        succ_probs /= succ_probs.sum()
        next_state = random.choices(successors, succ_probs)[0]

        mean_dur = G[state][next_state].get("mean_duration", 20)
        if target_color=="rosso":
            mean_dur *= 1.4
        dur = np.clip(np.random.normal(mean_dur, mean_dur*0.5), 5, 180)

        gmin, gmax = glucose_ranges[next_state]
        glucose_value = np.random.uniform(gmin, gmax)
        if noise_mode:
            glucose_value += np.random.normal(0,10)
            glucose_value = np.clip(glucose_value, 40, 400)

        seq.append((next_state, int(dur), round(glucose_value,1)))
        total_gluc_area += glucose_value * dur
        total_gluc_time += dur
        counts[next_state] += 1
        state = next_state
        elapsed += dur

    # ======================================================
    # 5. Calcolo GMI
    # ======================================================
    avg_gluc = total_gluc_area / total_gluc_time
    gmi = 3.31 + 0.02392 * avg_gluc
    if gmi < 7: gmi_color = "🟢 Verde"
    elif gmi < 8: gmi_color = "🟡 Giallo"
    else: gmi_color = "🔴 Rosso"

    # ======================================================
    # 6. Colore reale tramite algoritmo
    # ======================================================
    true_color, _ = analyze_simulate_sequence_compact(seq)

    # ======================================================
    # 7. Forza coerenza col target
    # ======================================================
    if target_color=="rosso" and "🔴" not in true_color:
        seq.append(("extremely_high",150,340))
    elif target_color=="giallo" and "🟡" not in true_color:
        seq.append(("high",80,220))
    elif target_color=="verde" and "🟢" not in true_color:
        seq.append(("normal",200,120))

    true_color,_ = analyze_simulate_sequence_compact(seq)

    return seq, true_color, gmi, avg_gluc, gmi_color

def simulate_sequence_with_duration(G, start="normal", steps=100, step_minutes=5):
    """
    Simula sequenze Markov con durate realistiche e step_minutes per letture.
    - Evita transizioni duplicate consecutive (normal->normal non viene generato)
    - Le letture successive dello stesso stato sono aggregate
    - Le glicemie cambiano gradualmente con random walk realistico
    """
    state = start
    seq = []

    glucose_ranges = {
        "extremely_low": (40, 55),
        "low": (56, 80),
        "normal": (81, 179),
        "high": (180, 249),
        "extremely_high": (250, 350)
    }

    prev_glucose = np.random.uniform(*glucose_ranges[state])
    prev_state_in_seq = None

    for _ in range(steps):
        # Successori diversi dallo stato corrente
        successors = [s for s in G.successors(state) if s != state]
        if not successors:
            successors = [state]  # fallback: resta nello stesso stato

        probs = [G[state][s]["prob"] for s in successors]
        next_state = random.choices(successors, probs)[0]

        # Determina durata dello stato
        data = G[state][next_state]
        if "mean_duration" in data:
            mean_dur = data["mean_duration"]
            dur = max(step_minutes, np.random.normal(mean_dur, mean_dur * 0.3))
        elif "durations" in data and len(data["durations"]) > 0:
            dur = random.choice(data["durations"])
        else:
            dur = random.uniform(5, 15)

        # Numero di letture in step_minutes
        num_steps = max(1, int(dur / step_minutes))
        gmin, gmax = glucose_ranges[next_state]

        # Random walk e aggregazione letture consecutive
        for _ in range(num_steps):
            delta = np.random.uniform(-2, 2)  # variazione graduale
            gluc = max(gmin, min(gmax, prev_glucose + delta))

            if prev_state_in_seq == next_state and len(seq) > 0:
                last_state, last_dur, last_gluc = seq[-1]
                seq[-1] = (last_state, last_dur + step_minutes, gluc)
            else:
                seq.append((next_state, step_minutes, round(gluc, 0)))
                prev_state_in_seq = next_state

            prev_glucose = gluc

        state = next_state

    return seq

In [ ]:
# --- Funzione per estrarre n-grammi con durate ---
def extract_ngrams(seq, n=3):
    ngrams = defaultdict(lambda: {'count': 0, 'durations': []})
    for i in range(len(seq) - n + 1):
        labels = tuple([s[0] for s in seq[i:i + n]])
        durations = [s[1] for s in seq[i:i + n]]
        ngrams[labels]['count'] += 1
        ngrams[labels]['durations'].append(durations)
    return ngrams


# --- Analisi pattern frequenti e rari ---
def analyze_patterns_labels(seq, n=3, top_k=5):
    ngrams = defaultdict(int)
    for i in range(len(seq) - n + 1):
        labels = tuple([s[0] for s in seq[i:i + n]])
        ngrams[labels] += 1

    results_sorted = sorted(ngrams.items(), key=lambda x: x[1], reverse=True)

    print(f"\nTop {top_k} pattern più frequenti:")
    for labels, count in results_sorted[:top_k]:
        print(f"Pattern {labels} | count={count}")

    print(f"\nTop {top_k} pattern più rari:")
    for labels, count in results_sorted[-top_k:]:
        print(f"Pattern {labels} | count={count}")


# --- Grafico top pattern frequenti ---
def plot_top_patterns(seq, n=3, top_k=10):
    ngrams = defaultdict(int)
    for i in range(len(seq) - n + 1):
        labels = tuple([s[0] for s in seq[i:i + n]])
        ngrams[labels] += 1

    results_sorted = sorted(ngrams.items(), key=lambda x: x[1], reverse=True)[:top_k]

    labels = [' → '.join(r[0]) for r in results_sorted]
    counts = [r[1] for r in results_sorted]

    plt.figure(figsize=(10, 5))
    plt.barh(labels, counts, color='skyblue')
    plt.xlabel("Frequenza")
    plt.ylabel("Pattern (n-grammi)")
    plt.title(f"Top {top_k} pattern più frequenti (n={n})")
    plt.gca().invert_yaxis()
    plt.show()


# --- Grafico durata media dei pattern ---
def plot_mean_critical_duration_patterns(seq, n=3, top_k=10):
    critical_states = {"high", "low", "extremely_high", "extremely_low"}
    ngrams = extract_ngrams(seq, n=n)
    results = []

    for labels, data in ngrams.items():
        # calcola durata solo degli stati critici
        critical_durations = []
        for d in data['durations']:
            dur = sum(d[i] for i, s in enumerate(labels) if s in critical_states)
            critical_durations.append(dur)
        mean_critical = sum(critical_durations) / len(critical_durations)
        results.append((' → '.join(labels), mean_critical, data['count']))

    results_sorted = sorted(results, key=lambda x: x[2], reverse=True)[:top_k]

    plt.figure(figsize=(10, 5))
    plt.barh([r[0] for r in results_sorted], [r[1] for r in results_sorted], color='salmon')
    plt.xlabel("Durata media degli stati critici (min)")
    plt.title(f"Durata media critica dei top {top_k} pattern (n={n})")
    plt.gca().invert_yaxis()
    plt.show()


# --- Visualizzazione grafo probabilistico ---
def plot_probabilistic_graph(G):
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000, arrows=True)
    edge_labels = {(u, v): f"{data['prob']:.2f}" for u, v, data in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.title("Modello probabilistico delle transizioni glicemiche")
    plt.show()


# sim_seq = sequenza simulata con durate
sim_seq = simulate_sequence_with_duration(G, start="normal", steps=5000)

print("\nSequenza simulata:", sim_seq)

# 1 Analisi pattern frequenti e rari
analyze_patterns_labels(sim_seq, n=4, top_k=5)

# 2 Grafico pattern più frequenti
plot_top_patterns(sim_seq, n=4, top_k=8)

# 3 Grafico durata media dei pattern
plot_mean_critical_duration_patterns(sim_seq, n=4, top_k=8)

# 4 Visualizzazione grafo probabilistico
#plot_probabilistic_graph(G)


## Confronto empirico tra dati reali e simulati


In [ ]:
def state_distribution_comparison(real_seq, sim_seq):
    # Conta le frequenze
    def calc_dist(seq):
        states = [s[0] for s in seq]
        counts = Counter(states)
        total = sum(counts.values())
        return {state: (count, count / total * 100) for state, count in counts.items()}, total

    real_dist, real_total = calc_dist(real_seq)
    sim_dist, sim_total = calc_dist(sim_seq)

    all_states = sorted(set(real_dist.keys()) | set(sim_dist.keys()))

    print("\n=== Confronto Distribuzione Stati ===")
    print(f"{'Stato':>10} | {'Reale (#)':>10} {'Reale (%)':>10} | {'Simulato (#)':>12} {'Simulato (%)':>12}")
    print("-" * 62)
    for state in all_states:
        real_count, real_perc = real_dist.get(state, (0, 0))
        sim_count, sim_perc = sim_dist.get(state, (0, 0))
        print(f"{state:>10} | {real_count:10d} {real_perc:10.2f}% | {sim_count:12d} {sim_perc:12.2f}%")
    print("-" * 62)
    print(f"{'Totale':>10} | {real_total:10d}{'':>11} | {sim_total:12d}")


def transition_distribution_comparison(real_seq, sim_seq):
    def calc_dist(seq):
        transitions = [(seq[i][0], seq[i + 1][0]) for i in range(len(seq) - 1)]
        counts = Counter(transitions)
        total = sum(counts.values())
        return {t: (c, c / total * 100) for t, c in counts.items()}, total

    real_dist, real_total = calc_dist(real_seq)
    sim_dist, sim_total = calc_dist(sim_seq)

    all_transitions = sorted(set(real_dist.keys()) | set(sim_dist.keys()))

    print("\n=== Confronto Distribuzione Transizioni ===")
    print(f"{'Transizione':>20} | {'Reale (#)':>10} {'Reale (%)':>10} | {'Simulato (#)':>12} {'Simulato (%)':>12}")
    print("-" * 78)
    for t in all_transitions:
        real_count, real_perc = real_dist.get(t, (0, 0))
        sim_count, sim_perc = sim_dist.get(t, (0, 0))
        tr_label = f"{t[0]} → {t[1]}"
        print(f"{tr_label:>20} | {real_count:10d} {real_perc:10.2f}% | {sim_count:12d} {sim_perc:12.2f}%")
    print("-" * 78)
    print(f"{'Totale':>20} | {real_total:10d}{'':>11} | {sim_total:12d}")


def average_duration_per_state(real_seq, sim_seq):
    # Calcolo media per stato, gestendo tuple di 2 o 3 elementi
    def calc_avg(seq):
        durations_per_state = defaultdict(list)
        for item in seq:
            if len(item) == 3:
                state, dur, _ = item
            elif len(item) == 2:
                state, dur = item
            else:
                raise ValueError(f"Unexpected tuple length {len(item)} in sequence: {item}")
            durations_per_state[state].append(dur)
        avg_per_state = {state: sum(durs) / len(durs) for state, durs in durations_per_state.items()}
        return avg_per_state

    real_avg = calc_avg(real_seq)
    sim_avg = calc_avg(sim_seq)

    all_states = sorted(set(real_avg.keys()) | set(sim_avg.keys()))

    print("\n=== Durata media per stato (minuti) ===")
    print(f"{'Stato':>15} | {'Reale':>10} | {'Simulato':>10}")
    print("-" * 40)
    for state in all_states:
        real_dur = real_avg.get(state, 0)
        sim_dur = sim_avg.get(state, 0)
        print(f"{state:>15} | {real_dur:10.2f} | {sim_dur:10.2f}")
    print("-" * 40)

def plot_transition_heatmap(real_seq, sim_seq, normalize=True):
    """
    Visualizza una heatmap delle transizioni di stato per sequenze reali e simulate.

    Parametri:
    - real_seq, sim_seq: liste di tuple (state, duration) o (state, duration, glucose)
    - normalize: se True, mostra frequenze relative (%)
    """

    def get_transitions(seq):
        trans = Counter()
        for i in range(len(seq) - 1):
            s1 = seq[i][0]
            s2 = seq[i + 1][0]
            trans[(s1, s2)] += 1
        return trans

    # Transizioni
    real_trans = get_transitions(real_seq)
    sim_trans = get_transitions(sim_seq)

    # Tutti gli stati coinvolti
    all_states = sorted(set([s for s, _ in real_trans.keys()] + [s for _, s in real_trans.keys()] +
                            [s for s, _ in sim_trans.keys()] + [s for _, s in sim_trans.keys()]))

    # Matrice reale
    real_matrix = [[real_trans.get((i, j), 0) for j in all_states] for i in all_states]
    sim_matrix = [[sim_trans.get((i, j), 0) for j in all_states] for i in all_states]

    if normalize:
        # Frequenze relative
        total_real = sum(real_trans.values())
        total_sim = sum(sim_trans.values())
        real_matrix = [[v / total_real * 100 for v in row] for row in real_matrix]
        sim_matrix = [[v / total_sim * 100 for v in row] for row in sim_matrix]

    # Heatmap reale
    plt.figure(figsize=(10, 5))
    sns.heatmap(real_matrix, annot=True, fmt=".1f" if normalize else "d",
                xticklabels=all_states, yticklabels=all_states, cmap="YlOrRd")
    plt.title("Transizioni - Reale")
    plt.xlabel("Stato successivo")
    plt.ylabel("Stato attuale")
    plt.show()

    # Heatmap simulata
    plt.figure(figsize=(10, 5))
    sns.heatmap(sim_matrix, annot=True, fmt=".1f" if normalize else "d",
                xticklabels=all_states, yticklabels=all_states, cmap="YlOrRd")
    plt.title("Transizioni - Simulato")
    plt.xlabel("Stato successivo")
    plt.ylabel("Stato attuale")
    plt.show()


# Esempio d'uso
state_distribution_comparison(seq, sim_seq)
transition_distribution_comparison(seq, sim_seq)
average_duration_per_state(seq, sim_seq)
plot_transition_heatmap(seq, sim_seq, normalize=True)


## Algoritmo per assegnare soglie alle squenze

In [ ]:
THRESHOLDS_DURATION = {
    "high": 90,
    "low": 30,
    "extremely_high": 45,
    "extremely_low": 10,
}


def normalize_target(target):
    """
    Rimuove emoji / caratteri non alfabetici e converte in minuscolo.
    """
    import re
    t = re.sub(r'[^\w]', '', str(target))
    return t.lower()


# ============================================================
# ANALISI COMPATTA SEQUENZA (target sintetico)
# ============================================================
def analyze_simulate_sequence_compact(seq, analize_day=None):
    """
    Classificazione clinica+adattiva basata su:
    - TIR/TBR/TAR (percentuale tempo nei range)
    - stabilità (oscillazioni e transizioni)
    - intensità dei picchi (extreme high/low)
    - variabilità (GV)
    Restituisce (status, score_anomalie)
    """

    # ============================================
    # 1. Determina durata analisi (flex)
    # ============================================
    total_minutes = sum(item[1] for item in seq)
    if analize_day is None:
        analize_day = max(1, round(total_minutes / 1440))
    max_minutes = analize_day * 1440

    # ============================================
    # 2. Ricostruzione timeline continua
    # ============================================
    times = []
    gl_values = []

    elapsed = 0
    for (state, dur, glucose) in seq:
        for _ in range(int(dur)):  # 1 minuto per punto
            if elapsed >= max_minutes:
                break
            times.append(elapsed)
            gl_values.append(glucose)
            elapsed += 1
        if elapsed >= max_minutes:
            break

    gl_values = np.array(gl_values)

    if len(gl_values) < 10:
        return "🟢 Verde", 0

    # ============================================
    # 3. Calcolo TIR/TBR/TAR secondo standard AGP
    # ============================================
    TBR1 = np.mean(gl_values < 70) * 100
    TBR2 = np.mean(gl_values < 54) * 100
    TIR  = np.mean((gl_values >= 70) & (gl_values <= 180)) * 100
    TAR1 = np.mean(gl_values > 180) * 100
    TAR2 = np.mean(gl_values > 250) * 100

    # Variabilità
    GV = np.std(gl_values)

    # ============================================
    # 4. Oscillazioni / transizioni instabili
    # ============================================
    # calco con differenze minute-to-minute
    delta = np.abs(np.diff(gl_values))
    fast_swings = np.sum(delta > 40)   # spike >40 mg/dl in 1 min
    big_swings  = np.sum(delta > 80)   # spike estremi

    # ============================================
    # 5. SCORE ANOMALIE (clinico + dinamico)
    # ============================================

    anomaly_score = 0

    # --- TBR (ipoglicemia) ---
    if TBR1 > 4:  anomaly_score += 2
    if TBR1 > 10: anomaly_score += 3
    if TBR2 > 1:  anomaly_score += 3   # ipoglicemia severa

    # --- TIR (range) ---
    if TIR < 70: anomaly_score += 1
    if TIR < 55: anomaly_score += 2
    if TIR < 45: anomaly_score += 3

    # --- TAR (iperglicemia) ---
    if TAR1 > 25: anomaly_score += 1
    if TAR1 > 40: anomaly_score += 2
    if TAR2 > 10: anomaly_score += 2

    # --- Variabilità ---
    if GV > 50: anomaly_score += 1
    if GV > 70: anomaly_score += 2
    if GV > 90: anomaly_score += 3

    # --- Oscillazioni ---
    if fast_swings > 20: anomaly_score += 1
    if fast_swings > 50: anomaly_score += 2
    if big_swings  > 15: anomaly_score += 3

    # ============================================
    # 6. Classificazione finale
    # ============================================

    if anomaly_score <= 3:
        return "🟢 Verde", anomaly_score
    elif anomaly_score <= 7:
        return "🟡 Giallo", anomaly_score
    else:
        return "🔴 Rosso", anomaly_score

def analyze_simulate_sequence_compact_v2(seq, analize_day=None):
    total_minutes = sum(item[1] for item in seq)
    if analize_day is None: analize_day = max(1, round(total_minutes / 1440))
    max_minutes = analize_day * 1440
    gl_values = []
    elapsed = 0
    for (state, dur, glucose) in seq:
        for _ in range(int(dur)):
            if elapsed >= max_minutes: break
            gl_values.append(glucose)
            elapsed += 1
        if elapsed >= max_minutes: break
    gl_values = np.array(gl_values)
    if len(gl_values) < 10: return " Verde", {}

    TBR1 = np.mean(gl_values < 70)*100
    TIR  = np.mean((gl_values >= 70)&(gl_values <= 180))*100
    TAR1 = np.mean(gl_values > 180)*100
    GV   = np.std(gl_values)

    status = " Verde"
    if (TIR < 55) or (TBR1 > 10) or (TAR1 > 40) or (GV > 70): status = "🔴 Rosso"
    elif (TIR < 70) or (TBR1 > 4) or (TAR1 > 25) or (GV > 50): status = "🟡 Giallo"
    return status, {}


def plot_sequence(seq, index=None, status=None):
    """
    Crea un grafico per visualizzare l'evoluzione dello stato nel tempo.
    Compatibile con sequenze (state, duration) o (state, duration, glucose_value)
    """
    state_map = {
        "extremely_low": -2,
        "low": -1,
        "normal": 0,
        "high": 1,
        "extremely_high": 2
    }
    color_map = {
        -2: "#0044cc",  # blu
        -1: "#00aaff",  # azzurro
        0: "#00cc44",  # verde
        1: "#ffcc00",  # giallo
        2: "#ff3300"  # rosso
    }

    times = [0]
    values = [state_map[seq[0][0]] if seq else 0]

    for item in seq:
        state = item[0]
        duration = item[1]
        times.append(times[-1] + duration)
        values.append(state_map[state])

    plt.figure(figsize=(10, 3))
    for i in range(1, len(times)):
        plt.plot(times[i - 1:i + 1], values[i - 1:i + 1], color=color_map[values[i]], linewidth=2)
    plt.yticks([-2, -1, 0, 1, 2], ["ext_low", "low", "normal", "high", "ext_high"])
    plt.xlabel("Tempo (minuti)")
    plt.title(f"Sequenza {index} – Stato finale: {status}")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


def generate_and_test_models_compact(number_models=10, total_days=7, show_graph=True):
    total_minutes = total_days * 1440  # conversione giorni → minuti
    results = []

    for i in range(number_models):
        seq, target, gmi, avg_gluc, gmi_color = simulate_random_sequence_realistic(G, total_minutes)
        status, anomalies = analyze_simulate_sequence_compact_v2(seq)
        results.append((status, anomalies, target))

        print(f"Modello {i}: {status}, anomalie: {anomalies}, target: {target}")

        if show_graph:
            plot_sequence(seq, index=i, status=status)

    status_counts = Counter([status for status, _, _ in results])
    anomalies_list = [anomalies for _, anomalies, _ in results]

    summary = {
        "status_counts": status_counts,
        "min_anomalies": min(anomalies_list),
        "max_anomalies": max(anomalies_list),
        "avg_anomalies": sum(anomalies_list) / len(anomalies_list)
    }

    print("\n" + "=" * 60)
    print(" RIEPILOGO DEI MODELLI ".center(60))
    print("=" * 60)
    for s, c in summary["status_counts"].items():
        perc = (c / len(results)) * 100
        print(f"{s}: {c} modelli ({perc:.1f}%)")
    print("-" * 60)
    print(f"Anomalie - min: {summary['min_anomalies']}, "
          f"max: {summary['max_anomalies']}, "
          f"media: {summary['avg_anomalies']:.2f}")
    print("=" * 60)

    return results, summary

## CODICE DI PROVA PER GENERARE ED ETICHETTARE MODELLI

In [ ]:

results, summary = generate_and_test_models_compact(number_models=3, total_days=10)

for i, (status, anomalies, target) in enumerate(results):
    print(f"Modello {i}: {status}, anomalie: {anomalies}, target simulato: {target}")

print("\n" + "=" * 60)
print(" RIEPILOGO DEI MODELLI".center(60))
print("=" * 60)

for s, c in summary["status_counts"].items():
    perc = (c / len(results)) * 100
    print(f"{s}: {c} modelli ({perc:.1f}%)")

print("-" * 60)
print(f"Anomalie - min: {summary['min_anomalies']}, "
      f"max: {summary['max_anomalies']}, "
      f"media: {summary['avg_anomalies']:.2f}")

if "mode_counts" in summary:
    print("\nModalità globali simulate:")
    for m, c in summary["mode_counts"].items():
        print(f"{m}: {c} modelli ({c / len(results) * 100:.1f}%)")

print("=" * 60)

## ESTRAZIONE FEATURES DA SEQUENZE + GENERAZIONE DI SEQUENZE + GENERAZIONE CSV

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd

def extract_features(sequence, max_minutes=None):
    """
    Estrazione ~50 feature numeriche robuste da sequenze CGM.
    Include:
    - Percentuali, run-length (mean, max, median, count)
    - Entropia stati
    - Glucose stats (avg, std, p10, p25, p50, p75, p90)
    - Weighted glucose per stato
    - Transizioni critiche e rapporti
    """
    import numpy as np

    states = ["extremely_low", "low", "normal", "high", "extremely_high"]
    counts = {s:0 for s in states}
    durations = {s:0 for s in states}
    run_lengths = {s:[] for s in states}
    glucose_values = []
    glucose_per_state = {s:[] for s in states}

    elapsed = 0
    current_run = None
    current_run_len = 0
    prev_state = None

    # --------------------------
    # SCANSIONE SEQUENZA
    # --------------------------
    for item in sequence:
        if len(item)==3:
            state, dur, glucose = item
        else:
            state, dur = item
            glucose = None

        if state not in states:
            state = "normal"
        if dur is None or dur <= 0 or np.isnan(dur) or np.isinf(dur):
            dur = 1
        if glucose is None or np.isnan(glucose) or np.isinf(glucose):
            glucose = 0

        glucose_values.append(glucose)
        glucose_per_state[state].append((glucose, dur))

        if max_minutes:
            elapsed += dur
            if elapsed > max_minutes:
                break

        counts[state] += 1
        durations[state] += dur

        if current_run is None:
            current_run = state
            current_run_len = dur
        else:
            if state == current_run:
                current_run_len += dur
            else:
                run_lengths[current_run].append(current_run_len)
                current_run = state
                current_run_len = dur

        prev_state = state

    if current_run is not None:
        run_lengths[current_run].append(current_run_len)

    # --------------------------
    # BUILD FEATURES
    # --------------------------
    features = {}
    total_time = sum(durations.values()) or 1

    # Percentuali e run-length
    for s in states:
        rl = run_lengths[s]
        features[f"perc_time_{s}"] = durations[s]/total_time
        features[f"run_mean_{s}"] = float(np.mean(rl)) if rl else 0
        features[f"run_max_{s}"] = float(np.max(rl)) if rl else 0
        features[f"run_median_{s}"] = float(np.median(rl)) if rl else 0
        features[f"run_count_{s}"] = len(rl)

    # Entropia stati
    counts_arr = np.array([counts[s] for s in states])
    if counts_arr.sum() > 0:
        p = counts_arr / counts_arr.sum()
        features["state_entropy"] = float(-np.sum(p * np.log2(p + 1e-12)))
    else:
        features["state_entropy"] = 0

    # Glucose stats globali
    if glucose_values:
        g = np.array(glucose_values)
        features["avg_glucose"] = float(np.mean(g))
        features["std_glucose"] = float(np.std(g))
        features["p10_glucose"] = float(np.percentile(g,10))
        features["p25_glucose"] = float(np.percentile(g,25))
        features["p50_glucose"] = float(np.percentile(g,50))
        features["p75_glucose"] = float(np.percentile(g,75))
        features["p90_glucose"] = float(np.percentile(g,90))
    else:
        for key in ["avg_glucose","std_glucose","p10_glucose","p25_glucose","p50_glucose","p75_glucose","p90_glucose"]:
            features[key] = 0

    # Glucose ponderata per stato (media ponderata durata)
    for s in states:
        vals = glucose_per_state[s]
        if vals:
            weighted_avg = np.sum([v*d for v,d in vals])/sum([d for _,d in vals])
        else:
            weighted_avg = 0
        features[f"weighted_glucose_{s}"] = weighted_avg

    # Transizioni critiche e rapporti
    features["ratio_high_to_extreme"] = counts["high"]/(counts["extremely_high"]+1) if counts["extremely_high"]>0 else 0
    features["ratio_low_to_extreme"] = counts["low"]/(counts["extremely_low"]+1) if counts["extremely_low"]>0 else 0
    features["ratio_normal_to_high"] = counts["normal"]/(counts["high"]+1) if counts["high"]>0 else 0
    features["ratio_low_to_high"] = counts["low"]/(counts["high"]+1) if counts["high"]>0 else 0
    features["ratio_extreme_low_to_extreme_high"] = counts["extremely_low"]/(counts["extremely_high"]+1) if counts["extremely_high"]>0 else 0
    features["ratio_high_to_normal"] = counts["high"]/(counts["normal"]+1) if counts["normal"]>0 else 0
    features["ratio_high_to_low"] = counts["high"]/(counts["low"]+1) if counts["low"]>0 else 0
    features["ratio_extreme_high_to_extreme_low"] = counts["extremely_high"]/(counts["extremely_low"]+1) if counts["extremely_low"]>0 else 0

    # Remove NaN/inf
    for k,v in features.items():
        if v is None or (isinstance(v,float) and (np.isnan(v) or np.isinf(v))):
            features[k] = 0

    return features

def save_intervals_csv(seq, filename, start_time="2024-02-13 00:01:32"):
    """
    Salva la sequenza in CSV con colonne:
    Start time, End time, Event, Duration
    Duration in formato '0 days hh:mm:ss' come da esempio.
    """
    import os
    from datetime import datetime, timedelta
    import pandas as pd

    if os.path.exists(filename):
        return  # già salvato

    os.makedirs(os.path.dirname(filename), exist_ok=True)
    current_time = datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S")
    rows = []

    for item in seq:
        # Supporta sequenze (state, duration) o (state, duration, glucose)
        state, duration = item[:2]

        start_dt = current_time
        end_dt = start_dt + timedelta(minutes=duration)
        duration_td = end_dt - start_dt  # timedelta → '0 days hh:mm:ss'

        rows.append({
            "Start time": start_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "End time": end_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "Event": state,
            "Duration": duration_td
        })

        current_time = end_dt

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"[OK] CSV salvato → {filename}")

def save_intervals_epoch_simple(seq, filename, start_time="2024-02-13 00:01:32"):
    if os.path.exists(filename):
        return  # già salvato

    os.makedirs(os.path.dirname(filename), exist_ok=True)
    start_dt = datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S")
    current_epoch = int(start_dt.timestamp())
    rows = []

    for state, duration, glucose in seq:
        start_e = current_epoch
        end_e = current_epoch + int(duration * 60)
        rows.append({
            "start_time": start_e,
            "end_time": end_e,
            "label": state
        })
        current_epoch = end_e

    pd.DataFrame(rows).to_csv(filename, index=False)
    print(f"[OK] Epoch-simple CSV → {filename}")



In [ ]:
import os
import joblib

def safe_save(obj, path):
    temp = path + ".tmp"
    joblib.dump(obj, temp)
    os.replace(temp, path)
# ============================================================
# GENERAZIONE DATASET CON PROPORZIONI TARGET FISSE (PROGRESSO + LOAD/SAVE)
# ============================================================
def generate_dataset_proportions(G, N=10000, total_minutes=10000, proportions=None):
    """
    Genera un dataset di sequenze CGM rispettando le proporzioni target desiderate.

    Parametri:
    - G: grafo degli stati
    - N: numero di sequenze totali
    - total_minutes: durata sequenza
    - proportions: dizionario con proporzioni target {"verde":0.6,"giallo":0.3,"rosso":0.1}

    Return:
    - pd.DataFrame con colonne: sequence, target, gmi, avg_gluc, gmi_color
    """
    import random
    from collections import defaultdict
    from tqdm.auto import tqdm

    if proportions is None:
        proportions = {"verde": 0.6, "giallo": 0.3, "rosso": 0.1}

    dataset = []
    counts_target = {k: 0 for k in proportions}

    for _ in tqdm(range(N), desc="Generazione sequenze"):
        # Decido target basato sulle quote residue
        remaining = {k: proportions[k]*N - counts_target[k] for k in proportions}
        possible_targets = [k for k,v in remaining.items() if v>0]
        if not possible_targets:
            break
        target_color = random.choice(possible_targets)

        # Genera sequenza con target forzato
        seq, _, gmi, avg_gluc, gmi_color = simulate_random_sequence_realistic(
            G,
            total_minutes=total_minutes,
            forced_target=target_color
        )

        dataset.append({
            "sequence": seq,
            "target": target_color,
            "gmi": gmi,
            "avg_gluc": avg_gluc,
            "gmi_color": gmi_color
        })
        counts_target[target_color] += 1

    df = pd.DataFrame(dataset)

    # Stampa distribuzione finale
    print("\nDistribuzione target finale:")
    counts = df['target'].value_counts()
    for k,v in counts.items():
        print(f"  {k}: {v} sequenze ({v/len(df)*100:.2f}%)")

    return df

# ============================================================
# GENERA E SALVA I 4 DATASET CON PROPORZIONI TARGET FISSE + LOAD IF EXISTS
# ============================================================
def generate_and_save_all_datasets(G, N_dict=None, total_minutes_per_day=1440, cache_dir="datasets"):
    """
    Genera dataset CGM per 15, 30, 60, 90 giorni con proporzioni target fisse
    e salva in cache_dir. Se il dataset esiste già lo carica.
    """
    import os
    os.makedirs(cache_dir, exist_ok=True)

    if N_dict is None:
        N_dict = {15: 30000, 30: 15000, 60: 8000, 90: 5000}

    proportions = {"verde": 0.6, "giallo": 0.3, "rosso": 0.1}

    for days, N in N_dict.items():
        path = os.path.join(cache_dir, f"dataset_{days}d.pkl")

        if os.path.exists(path):
            try:
                df = joblib.load(path)
                print(f"\n✔ Dataset {days} giorni trovato e caricato da {path}")
                counts = df['target'].value_counts()
                print("Distribuzione target già presente:")
                for k,v in counts.items():
                    print(f"  {k}: {v} sequenze ({v/len(df)*100:.2f}%)")
                continue
            except:
                print(f"\n⚠ Dataset {days} giorni corrotto, lo rigenero…")

        print(f"\n==============================")
        print(f" Generazione dataset {days} giorni ({N} sequenze)")
        print(f"==============================")

        total_minutes = days * total_minutes_per_day
        df = generate_dataset_proportions(
            G,
            N=N,
            total_minutes=total_minutes,
            proportions=proportions
        )

        safe_save(df, path)
        print(f"✔ Dataset {days} giorni salvato in {path}")


generate_and_save_all_datasets(G)

In [ ]:
df = generate_dataset_proportions(G, N=500, total_minutes=15*24*60)
analyze_simulate_sequence_compact_v2(df['sequence'][9])
print(extract_features(df['sequence'][9]))

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from collections import Counter
from Levenshtein import distance as lev_distance

def sequences_to_state_strings(df):
    """Trasforma le sequenze in stringhe di stati per confronto."""
    seq_strings = []
    for seq in df['sequence']:
        states = [item[0] for item in seq]  # considera solo lo stato
        seq_strings.append(' '.join(states))
    return seq_strings

def analyze_sequence_similarity(df, sample_size=None):
    """Analizza quanto le sequenze sono simili tra loro."""
    seq_strings = sequences_to_state_strings(df)
    if sample_size is not None and len(seq_strings) > sample_size:
        seq_strings = np.random.choice(seq_strings, sample_size, replace=False)

    # Calcola quante sequenze sono identiche
    counts = Counter(seq_strings)
    num_duplicates = sum([c-1 for c in counts.values() if c>1])
    pct_duplicates = num_duplicates / len(seq_strings) * 100
    print(f"Sequenze duplicate: {num_duplicates}/{len(seq_strings)} ({pct_duplicates:.2f}%)")

    # Calcola distanza media di Levenshtein tra un campione di sequenze
    N = min(200, len(seq_strings))  # non fare tutte per velocità
    sample = np.random.choice(seq_strings, N, replace=False)
    distances = []
    for i in tqdm(range(len(sample))):
        for j in range(i+1, len(sample)):
            d = lev_distance(sample[i], sample[j])
            distances.append(d)
    distances = np.array(distances)
    print(f"Distanza media Levenshtein: {distances.mean():.2f}")
    print(f"Distanza min: {distances.min()}, max: {distances.max()}")
    return counts, distances


In [ ]:
df_15 = joblib.load("datasets/dataset_15d.pkl")
counts, distances = analyze_sequence_similarity(df_15)  # prende un campione per velocità

# UMAP + PCA

In [ ]:
from collections import Counter
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.sparse import csr_matrix
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


NGRAM_N = 3
TOP_K_NGRAMS = 3000
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
RANDOM_STATE = 42


# -----------------------------
# Funzioni ausiliarie
# -----------------------------
def discretize_duration_by_state(state, dur, thresholds):
    """
    Discretizza la durata di uno stato in breve (N) o lunga (L) rispetto alla soglia specifica.
    """
    thr = thresholds.get(state, 30)
    return "N" if dur < thr else "L"


def build_ngram_vocab(sequences, n=NGRAM_N, top_k=TOP_K_NGRAMS):
    """
    Costruisce il vocabolario dei n-gram più frequenti da una lista di sequenze.
    """
    counter = Counter()
    for seq in tqdm(sequences, desc="Conteggio n-gram"):
        if len(seq) < n:
            continue
        states = [s for s, _, _ in seq]
        durs = [discretize_duration_by_state(s, d, THRESHOLDS_DURATION) for s, d, _ in seq]
        # Genera n-gram e aggiorna contatore
        for i in range(len(seq) - n + 1):
            ng = "_".join(f"{st}({du})" for st, du in zip(states[i:i + n], durs[i:i + n]))
            counter[ng] += 1
    # Restituisce solo i top_k n-gram più frequenti
    return [ng for ng, _ in counter.most_common(top_k)]


def sequences_to_sparse_matrix(sequences, vocab, n=NGRAM_N):
    """
    Trasforma le sequenze in una matrice sparsa n-gram (conteggio frequenze).
    """
    vocab_index = {ng: i for i, ng in enumerate(vocab)}
    row_idx, col_idx, data = [], [], []

    for seq_id, seq in enumerate(tqdm(sequences, desc="Vettorizzazione")):
        if len(seq) < n:
            continue
        states = [s for s, _, _ in seq]
        durs = [discretize_duration_by_state(s, d, THRESHOLDS_DURATION) for s, d, _ in seq]
        counts = Counter()
        for i in range(len(seq) - n + 1):
            ng = "_".join(f"{st}({du})" for st, du in zip(states[i:i + n], durs[i:i + n]))
            if ng in vocab_index:
                counts[vocab_index[ng]] += 1
        for col, c in counts.items():
            row_idx.append(seq_id)
            col_idx.append(col)
            data.append(c)

    return csr_matrix((data, (row_idx, col_idx)), shape=(len(sequences), len(vocab)))


def analyze_distribution_by_target(df, n=NGRAM_N, top_k=TOP_K_NGRAMS):
    """
    Analizza la distribuzione delle sequenze per target usando UMAP e PCA.
    """
    if df.empty:
        print("Dataset vuoto.")
        return

    sequences = df["sequence"].tolist()
    print(f"Numero sequenze: {len(sequences)}")

    # 1. Costruzione vocabolario n-gram
    vocab = build_ngram_vocab(sequences, n=n, top_k=top_k)
    print(f"Vocabolario n-gram: {len(vocab)} termini")

    # 2. Creazione matrice sparsa
    X_sparse = sequences_to_sparse_matrix(sequences, vocab, n=n)
    print(f"Shape matrice sparsa: {X_sparse.shape}")

    # 3. Standardizzazione
    scaler = StandardScaler(with_mean=False)
    X_scaled = scaler.fit_transform(X_sparse)

    # 4. UMAP embedding
    print("Eseguo UMAP...")
    reducer = umap.UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        random_state=RANDOM_STATE
    )
    emb_umap = reducer.fit_transform(X_scaled)

    # 5. PCA embedding
    print("Eseguo PCA (2 componenti)...")
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    emb_pca = pca.fit_transform(X_scaled.toarray())
    explained_var = pca.explained_variance_ratio_.sum()
    print(f"Varianza spiegata PCA: {explained_var:.2%}")

    # 6. Palette dinamica
    palette = {}
    for t in df["target"].unique():
        if "rosso" in t.lower() or "🔴" in t:
            palette[t] = "red"
        elif "giallo" in t.lower() or "🟡" in t:
            palette[t] = "gold"
        elif "verde" in t.lower() or "🟢" in t:
            palette[t] = "green"

    # 7. Visualizzazione UMAP
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=emb_umap[:, 0], y=emb_umap[:, 1], hue=df["target"], palette=palette, s=40)
    plt.title("Distribuzione UMAP per target")
    plt.legend(title="Target")
    plt.show()

    # 8. Visualizzazione PCA
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=emb_pca[:, 0], y=emb_pca[:, 1], hue=df["target"], palette=palette, s=40)
    plt.title(f"Distribuzione PCA per target (Varianza spiegata: {explained_var:.2%})")
    plt.legend(title="Target")
    plt.show()

    return {"emb_umap": emb_umap, "emb_pca": emb_pca, "palette": palette}


results_all = analyze_distribution_by_target(df_long, n=3, top_k=3000)

# DETECTION PATTERN (N-GRAM)
-Globale su sequenze generale

-Specifio per csv paziente

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import Counter

# =========================================================
# FILTRO ANTI-ALTERNANZA
# =========================================================
def is_two_state_alternating(pattern):
    """
    Rileva se un pattern è una mera oscillazione tra due stati (es. A -> B -> A -> B).
    Pulisce eventuali annotazioni di durata, es. 'high(N)' diventa 'high'.
    Restituisce True se il pattern va scartato.
    """
    def clean_state(x):
        return x.split("(")[0] if isinstance(x, str) else x

    cleaned = [clean_state(p) for p in pattern]
    unique_states = list(set(cleaned))

    # Deve contenere esattamente due stati per essere un'alternanza banale
    if len(unique_states) != 2:
        return False

    # Verifica se l'alternanza è perfetta
    exp1 = [unique_states[i % 2] for i in range(len(cleaned))]
    exp2 = [unique_states[(i + 1) % 2] for i in range(len(cleaned))]

    return cleaned == exp1 or cleaned == exp2

# =========================================================
# FUNZIONI BASE DI UTILITÀ
# =========================================================
def csv_to_sequence(csv_path):
    """Converte un CSV di intervalli in una lista di tuple (Stato, Durata)."""
    df = pd.read_csv(csv_path)
    df["duration_minutes"] = pd.to_timedelta(df["Duration"]).dt.total_seconds() / 60
    return df[["Event", "duration_minutes"]].values.tolist()

def discretize_duration_by_state(state, duration, thresholds_dict=None):
    """Discretizza la durata in Breve (N) o Lunga (L) rispetto a soglie cliniche."""
    if not thresholds_dict or state not in thresholds_dict:
        return ""

    thr = thresholds_dict[state]
    if isinstance(thr, (list, tuple)):
        thr = thr[0]

    return "L" if duration > thr else "N"

def normalize_sequence(sequence):
    """Garantisce che ogni elemento della sequenza sia nel formato (Stato, Durata, Glucosio)."""
    seq_full = []
    for item in sequence:
        if isinstance(item, np.ndarray):
            item = item.tolist()

        if len(item) == 2:
            s, d = item
            seq_full.append((s, d, 0.0))  # Placeholder per il glucosio se mancante
        elif len(item) == 3:
            seq_full.append(tuple(item))
        else:
            raise ValueError(f"Formato sequenza non supportato: {item}")
    return seq_full

# =========================================================
# ESTRAZIONE N-GRAMMI E PATTERN
# =========================================================
def extract_ngrams(sequence, n_list=[4,5], use_duration=False, thresholds_dict=None):
    """Estrae n-grammi da una sequenza, includendo opzionalmente la durata discretizzata."""
    seq = normalize_sequence(sequence)
    states = [s for s, d, g in seq]

    durations = [
        discretize_duration_by_state(s, d, thresholds_dict) if use_duration else ""
        for s, d, g in seq
    ]

    ngrams = []
    for n in n_list:
        for i in range(len(seq) - n + 1):
            # Costruisce il pattern unendo Stato e Durata (es. 'high(L)') se richiesto
            ng = tuple(
                f"{st}({du})" if du else st
                for st, du in zip(states[i:i+n], durations[i:i+n])
            )
            ngrams.append(ng)

    return ngrams

# =========================================================
# CREAZIONE DATASET DEI PATTERN GLOBALI (LIFT ANALYSIS)
# =========================================================
def build_detection_patterns(
    df,
    n_list=[4,5],
    top_k=50,
    use_duration=False,
    thresholds_dict=None,
    cache_dir="pattern_cache",
    csv_dir="global_pattern"
):
    """Costruisce il dizionario globale dei pattern, calcolando Lift e target dominante."""
    os.makedirs(cache_dir, exist_ok=True)
    os.makedirs(csv_dir, exist_ok=True)

    hash_str = f"{hash(tuple(n_list))}_{top_k}_{use_duration}_{str(thresholds_dict)}"
    cache_path = os.path.join(cache_dir, f"detection_patterns_{abs(hash(hash_str))}.pkl")

    # Controlla la cache per evitare ricalcoli costosi
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            detection_patterns = pickle.load(f)
        global_csv_path = os.path.join(csv_dir, "global_final_pattern.csv")
        detection_patterns.to_csv(global_csv_path, index=False)
        print("✔ Pattern globali caricati dalla cache.")
        return detection_patterns

    all_patterns = []

    for n in n_list:
        print(f"\n--- Estrazione pattern globali per n={n} ---")
        targets = df["target"].unique()
        pattern_counts = {}
        total_counts = {}

        for target in targets:
            seqs = df[df["target"] == target]["sequence"].tolist()
            ngrams = []

            for seq in tqdm(seqs, desc=f"Target: {target}"):
                raw_ngrams = extract_ngrams(
                    seq,
                    n_list=[n],
                    use_duration=use_duration,
                    thresholds_dict=thresholds_dict
                )
                # Applica filtro anti-alternanza strumentale
                filtered = [ng for ng in raw_ngrams if not is_two_state_alternating(ng)]
                ngrams.extend(filtered)

            pattern_counts[target] = Counter(ngrams)
            total_counts[target] = sum(pattern_counts[target].values())

        # Costruisce il DataFrame per il parametro N corrente
        all_keys = set().union(*[c.keys() for c in pattern_counts.values()])

        rows = []
        for pat in all_keys:
            row = {"pattern": pat, "n": n}
            lift_vals = {}

            for target in targets:
                cnt = pattern_counts[target].get(pat, 0)
                freq_t = cnt / total_counts[target] if total_counts[target] else 0

                cnt_others = sum(pattern_counts[t].get(pat, 0) for t in targets if t != target)
                total_others = sum(total_counts[t] for t in targets if t != target)

                freq_o = cnt_others / total_others if total_others else 1e-9
                lift = freq_t / freq_o if freq_o > 0 else 0

                row[f"freq_{target}"] = freq_t
                row[f"lift_{target}"] = lift
                lift_vals[target] = lift

            # Identifica il target clinico dominante per questo pattern
            row["dominant_target"] = max(lift_vals, key=lift_vals.get)
            row["max_lift"] = lift_vals[row["dominant_target"]]
            rows.append(row)

        df_n = pd.DataFrame(rows).sort_values("max_lift", ascending=False).head(top_k)
        all_patterns.append(df_n)

        # Salvataggio CSV locale
        csv_path_n = os.path.join(csv_dir, f"patterns_globali_n{n}.csv")
        df_n.to_csv(csv_path_n, index=False)
        print(f"✔ CSV globale (n={n}) salvato in: {csv_path_n}")

    # Concatena tutti i risultati n-dimensionali
    detection_patterns = (
        pd.concat(all_patterns, ignore_index=True)
        .sort_values("max_lift", ascending=False)
        .reset_index(drop=True)
    )

    global_csv_path = os.path.join(csv_dir, "global_final_pattern.csv")
    detection_patterns.to_csv(global_csv_path, index=False)
    print(f"\n✔ Dataset aggregato salvato in: {global_csv_path}")

    # Salva in cache
    with open(cache_path, "wb") as f:
        pickle.dump(detection_patterns, f)

    return detection_patterns

# =========================================================
# VISUALIZZAZIONE E HEATMAP
# =========================================================
def plot_detection_patterns_heatmap(detection_patterns, targets, metric="freq", save_path=None):
    """Genera e (opzionalmente) salva la Heatmap per i pattern."""
    metric = metric.lower()
    columns = [f"{metric}_{t}" for t in targets]

    heatmap_data = detection_patterns[columns].copy()
    heatmap_data.index = detection_patterns["pattern"].astype(str)

    plt.figure(figsize=(12, max(4, len(heatmap_data) * 0.5)))
    sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': metric.upper()})
    plt.title(f"Distribuzione {metric.upper()} dei Top Pattern Glicemici", pad=15)
    plt.ylabel("Pattern N-gram (Sequenza di Stati)")

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"✔ Heatmap salvata per LaTeX in: {save_path}")

    plt.show()

# =========================================================
# INFERENZA SUL PAZIENTE: MATCH PATTERN GLOBALI
# =========================================================
def extract_patient_patterns(sequence, n_list=[4,5], top_k=20, use_duration=False, thresholds_dict=None):
    """Estrae e filtra i pattern frequenti di un singolo paziente."""
    raw = extract_ngrams(sequence, n_list=n_list, use_duration=use_duration, thresholds_dict=thresholds_dict)
    filtered = [ng for ng in raw if not is_two_state_alternating(ng)]

    freq = Counter(filtered)
    df = pd.DataFrame([{"pattern": k, "freq_paziente": v} for k, v in freq.items()])

    if df.empty:
        return pd.DataFrame()

    return df.sort_values("freq_paziente", ascending=False).head(top_k)


def patient_vs_global_patterns_from_df(df_patient, detection_patterns):
    """Assegna un punteggio di rischio al paziente confrontando i suoi pattern con il database globale."""
    if df_patient.empty:
        return df_patient

    df_patient = df_patient.copy()
    global_dict = detection_patterns.set_index("pattern").to_dict(orient="index")

    dom, freq_g, lift_g = [], [], []

    for pat in df_patient["pattern"]:
        if pat in global_dict:
            row = global_dict[pat]
            targ = row["dominant_target"]
            dom.append(targ)
            freq_g.append(row[f"freq_{targ}"])
            lift_g.append(row[f"lift_{targ}"])
        else:
            dom.append(None)
            freq_g.append(0.0)
            lift_g.append(0.0)

    df_patient["dominant_target_global"] = dom
    df_patient["freq_global"] = freq_g
    df_patient["lift_global"] = lift_g

    return df_patient

In [ ]:
#df_long = generate_dataset_proportions(G, N=30000, total_minutes=90*24*60)
df_long = joblib.load("../data/datasets/dataset_90d.pkl")
print("[INFO] BUILD PATTERN GLOBALI SENZA DURATA")
detection_patterns_no_duration = build_detection_patterns(df_long, n_list=[4], top_k=200, use_duration=False)
display(detection_patterns_no_duration)
plot_detection_patterns_heatmap(detection_patterns_no_duration, df_long["target"].unique(), metric="freq")
plot_detection_patterns_heatmap(detection_patterns_no_duration, df_long["target"].unique(), metric="lift")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

# ===== 1️⃣ Carica CSV =====
df = pd.read_csv("../data/global_pattern/global_final_pattern.csv")

df.columns = [
    "pattern",
    "count",
    "support_pattern",
    "support_target",
    "joint_support",
    "confidence",
    "expected_support",
    "lift_base",
    "risk_label",
    "lift_final"
]

# ===== 2️⃣ Converti pattern =====
df["pattern"] = df["pattern"].apply(ast.literal_eval)

# ===== 3️⃣ Prendi TOP 10 per Lift =====
df_top = df.sort_values("lift_final", ascending=False).head(10).reset_index(drop=True)

# ===== 4️⃣ Stati =====
states = ["extremely_low", "low", "normal", "high", "extremely_high"]

# ===== 5️⃣ Costruisci matrice =====
heatmap_data = []
pattern_labels = []

for _, row in df_top.iterrows():
    pattern = row["pattern"]
    lift = row["lift_final"]

    counts = [pattern.count(s) for s in states]
    weighted = [c * lift for c in counts]

    heatmap_data.append(weighted)
    pattern_labels.append(" → ".join(pattern))

heatmap_matrix = np.array(heatmap_data)

# ===== 6️⃣ Plot =====
plt.figure(figsize=(10, 6))
plt.imshow(heatmap_matrix, aspect='auto')
plt.colorbar()

plt.xticks(range(len(states)), states, rotation=45)
plt.yticks(range(len(pattern_labels)), pattern_labels)

plt.xlabel("Glycemic States")
plt.ylabel("Top 10 Patterns (by Lift)")
plt.title("Top 10 Global Patterns – Weighted by Lift")

plt.tight_layout()
plt.show()

ALTRO COSA NEL FILTRO N>=3 cosi RIMUOVIAMO I TIME SWING
